# HumanAI Detect - Ozellik Cikarimi + Held-out Yenileme (Claude/Sonnet-5 Topup, Colab GPU)

**Baglam:** Ana `human`/`ai_raw`/`ai_humanized` havuzlarina Claude Sonnet 5 (3. uretici) ile
uretilen takviye (209 ai_raw_anthropic + 209 ai_humanized_anthropic) yerelde birlestirildi
(bkz. proje notlari, 2026-07-20). Yeni toplam: human=3262, ai_raw=4319, ai_humanized=4315
(**11896 ornek**, onceki GPT-4o-mini-topup-sonrasi 11478'den +418).

**Neden Colab:** Ayni 2-gecisli akis, ayni sebep (`build_features.py` scaler/uzunluk-
residualizasyonunu sizintisiz yapabilmek icin `holdout_ids.txt`'e ihtiyac duyuyor, o da
`groups.parquet`'ten -- build_features'in kendi ciktisindan -- turetiliyor, yumurta-tavuk
sorunu). **Onemli:** embedding cache Colab oturumlari arasinda KALICI DEGIL (her oturum
sifirdan `git clone` + `/content` -- onceki GPT-4o-mini calistirmasindan kalan onbellek bu
oturumda mevcut degil), yani 1. gecis yine TUM 11896 ornegin embedding'ini sifirdan
hesaplayacak (~30-60 dk GPU'da, GPT-4o-mini topup'taki sureyle ayni buyukluk mertebesinde).

**Iki-gecisli akis:**
1. **1. gecis:** `build_features.py`'yi holdout OLMADAN calistir (gecici, tum veriyle fit).
2. `make_holdout_split.py` ile YENI (11896 orneği icine alan, Claude/Sonnet-5 dahil) `holdout_ids.txt`'i uret.
3. **2. gecis:** `build_features.py`'yi TEKRAR calistir -- artik holdout_ids.txt var, scaler/
   residualizer sadece train'den fit edilir (sizintisiz). Embedding'ler bu oturumda zaten
   1. gecisten onbellekte oldugu icin **bu ikinci gecis COK hizli** olacak (sadece
   stilometri+fusion, birkac dakika).

**Onemli -- neden holdout'u yeniden uretmek gerekiyor:** Eski `holdout_ids.txt` Claude/
Sonnet-5 orneklerinden hicbirini icermiyordu -- yani resmi held-out degerlendirmesi
Claude'a karsi capraz-uretici genellemeyi hic yansitmayacakti. Yeni holdout, tum
ureticilerden (Qwen, GPT-4o-mini, Claude/Sonnet-5) orantili orneklenmis olacak.

In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('UYARI: GPU yok! Runtime -> Change runtime type -> GPU (T4 yeterli) secip tekrar dene.')

In [ ]:
# 2. Repo klonla (guncel kod)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect
!git log --oneline -5

In [ ]:
# 3. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q

In [ ]:
# 4. .env olustur (bu adim icin API key gerekmez)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')

In [ ]:
# 5. Guncel (Claude/Sonnet-5 dahil, birlestirilmis) interim dosyalarini yerel makineden
#    yukle: human.jsonl (3262), ai_raw.jsonl (4319), ai_humanized.jsonl (4315) -- UCUNU BIRDEN sec
# Bu dosyalar buyuk (toplam ~900MB-1GB) -- yuklemesi birkac dakika surebilir.
from pathlib import Path
from google.colab import files

for label in ['human', 'ai_raw', 'ai_humanized']:
    Path(f'data/interim/{label}').mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    if 'ai_humanized' in name:
        Path(name).rename('data/interim/ai_humanized/ai_humanized.jsonl')
    elif 'ai_raw' in name:
        Path(name).rename('data/interim/ai_raw/ai_raw.jsonl')
    elif 'human' in name:
        Path(name).rename('data/interim/human/human.jsonl')

for label in ['human', 'ai_raw', 'ai_humanized']:
    p = Path(f'data/interim/{label}/{label}.jsonl')
    n = sum(1 for _ in open(p, encoding='utf-8')) if p.exists() else 0
    print(f'{label}: {n} ornek')

In [ ]:
# 6. 1. GECIS: build_features.py -- holdout_ids.txt YOK, tum veriyle gecici fit.
# Bu adim UZUN surer (11896 ornegin TAMAMININ BERTurk embedding'i sifirdan hesaplaniyor,
# bu Colab oturumunda hicbir gecerli onbellek olmadigi icin). GPU'da tahmini ~30-60 dk.
!python scripts/build_features.py

In [ ]:
# 7. YENI holdout_ids.txt uret (artik ai_raw_anthropic/ai_humanized_anthropic orneklerini de iceriyor)
!python scripts/make_holdout_split.py
!wc -l data/processed/holdout_ids.txt

In [ ]:
# 8. 2. GECIS: build_features.py'yi TEKRAR calistir -- artik holdout_ids.txt var, scaler/
# uzunluk-residualizasyonu SADECE train'den (sizintisiz) fit edilecek. Embedding'ler onbellekte
# oldugu icin bu gecis COK HIZLI olmali (sadece stilometri+fusion, birkac dakika).
#
# ONEMLI: build_features.py, stylometric/embeddings/fused ciktilarindan herhangi biri zaten
# VARSA atlar -- bu yuzden 2. gecisten once bunlari silmemiz, ama embedding CACHE'ini
# (data/processed/embedding_cache/) SILMEMEMIZ lazim (asil hiz kazanci ondan geliyor).
from pathlib import Path
for f in ['stylometric.parquet', 'fused.parquet', 'groups.parquet', 'length_residualizer.json',
          'embeddings_berturk.parquet']:
    p = Path('data/processed') / f
    if p.exists():
        p.unlink()
        print(f'silindi: {p}')

!python scripts/build_features.py

In [ ]:
# 9. Sonuclari yerel makineye indir
from pathlib import Path
from google.colab import files

for f in ['stylometric.parquet', 'embeddings_berturk.parquet', 'groups.parquet',
          'fused.parquet', 'holdout_ids.txt', 'length_residualizer.json']:
    src = Path('data/processed') / f
    if src.exists():
        print(f'indiriliyor: {f}')
        files.download(str(src))
    else:
        print(f'UYARI: {f} bulunamadi.')

## Yerel Makineye Aktarim

Yukaridaki hucre 6 dosyayi indirir: `stylometric.parquet`, `embeddings_berturk.parquet`,
`groups.parquet`, `fused.parquet`, `holdout_ids.txt`, `length_residualizer.json`.

Kendi makinende hepsini `data/processed/` altina (mevcutlarin UZERINE) koy, sonra bana
haber ver -- sonraki adim: `colab_measure_calibration.ipynb` ile (mevcut notebook, degisiklik
gerekmiyor) modeli yeniden egitip kalibre edecegim. **Not:** o notebook'ta "GPU runtime"
kullanma -- egitim adimi (sklearn/XGBoost/CatBoost) GPU kullanmiyor, saf CPU-bound; "CPU
+ Yuksek RAM" runtime tercih edilmeli.